In [ ]:

import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import numpy as np
import pandas as pd
import wfdb
from scipy import signal

In [ ]:
PTB_DIR = r"C:\Users\eq26\Desktop\ECG LM\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
PTB_PATH = os.path.join(PTB_DIR, "ptbxl_database.csv")

ptb_df = pd.read_csv(PTB_PATH)

In [ ]:
ptb_df.head()

In [ ]:
row = ptb_df.iloc[0]

# PTB-XL commonly stores the 500 Hz path in filename_hr
record_path = os.path.join(PTB_DIR, row["filename_hr"])

record = wfdb.rdsamp(record_path)
signal = record[0]          # shape usually [samples, leads]
meta = record[1]

print("signal shape:", signal.shape)
print("fs:", meta["fs"])

In [ ]:
import numpy as np
from scipy.signal import resample as scipy_resample

TARGET_FS   = 500
SEGMENT_SEC = 5
SEGMENT_LEN = TARGET_FS * SEGMENT_SEC
EPS = 1e-8


def ensure_lead_first(x, expected_leads=12):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D ECG array, got shape {x.shape}")
    if x.shape[0] == expected_leads:
        return x
    if x.shape[1] == expected_leads:
        return x.T
    raise ValueError(f"Could not infer lead dimension from shape {x.shape}")


def zscore_per_lead(x, eps=EPS):
    mean = x.mean(axis=1, keepdims=True)
    std  = x.std(axis=1, keepdims=True)
    return ((x - mean) / (std + eps)).astype(np.float32)


def preprocess_record(waveform, orig_fs, target_fs=TARGET_FS,
                       segment_len=SEGMENT_LEN):
    """
    Returns float32 array of shape [n_segments, 12, segment_len].
    Raises ValueError for bad records so the Dataset can retry.
    """
    x = ensure_lead_first(waveform)

    # NaN / Inf guard
    if not np.isfinite(x).all():
        bad_ratio = (~np.isfinite(x)).mean()
        if bad_ratio > 0.20:
            raise ValueError(f"Record has {bad_ratio:.0%} corrupt samples")
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    # Resample if needed
    if orig_fs != target_fs:
        target_len = int(x.shape[1] * target_fs / orig_fs)
        x = scipy_resample(x, target_len, axis=1).astype(np.float32)

    x = zscore_per_lead(x)

    n_seg = x.shape[1] // segment_len
    if n_seg == 0:
        raise ValueError("Record too short for even one segment")
    x = x[:, :n_seg * segment_len]
    segs = x.reshape(12, n_seg, segment_len)
    segs = np.transpose(segs, (1, 0, 2))  # [n_seg, 12, seg_len]

    # Final NaN check
    if not np.isfinite(segs).all():
        raise ValueError("NaN in segments after normalization")

    return segs.astype(np.float32)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_raw_ecg(signal, fs=500, title="Raw ECG"):
    """
    signal shape: (samples, leads) or (leads, samples)
    """
    x = np.asarray(signal)

    if x.shape[0] == 12 and x.shape[1] != 12:
        x = x.T  # convert to (samples, leads)

    n_samples, n_leads = x.shape
    t = np.arange(n_samples) / fs

    fig, axes = plt.subplots(n_leads, 1, figsize=(14, 18), sharex=True)

    lead_names = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

    for i in range(n_leads):
        axes[i].plot(t, x[:, i], linewidth=0.8)
        axes[i].set_ylabel(lead_names[i], rotation=0, labelpad=20)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_raw_ecg(signal, fs=meta["fs"], title="PTB-XL Raw 10s ECG")

In [ ]:
def plot_segment(segment, fs=500, title="Preprocessed 5s Segment"):
    """
    segment shape: (12, 2500)
    """
    x = np.asarray(segment)
    t = np.arange(x.shape[1]) / fs

    fig, axes = plt.subplots(x.shape[0], 1, figsize=(14, 18), sharex=True)
    lead_names = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

    for i in range(x.shape[0]):
        axes[i].plot(t, x[i], linewidth=0.8)
        axes[i].set_ylabel(lead_names[i], rotation=0, labelpad=20)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

## Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, channels=256, kernel_size=2, stride=2):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=kernel_size,
            stride=stride
        )
        self.norm = nn.GroupNorm(num_groups=1, num_channels=channels)  # layer-like norm over channels
        self.act = nn.GELU()

    def forward(self, x):
        # x: [B, C, T]
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        return x


class ECGFeatureExtractor(nn.Module):
    """
    4 CNN blocks.
    First block maps 12 leads -> 256 channels.
    Remaining 3 blocks keep 256 channels.
    """
    def __init__(self, in_channels=12, hidden_channels=256):
        super().__init__()

        self.input_proj = nn.Conv1d(
            in_channels=in_channels,
            out_channels=hidden_channels,
            kernel_size=2,
            stride=2
        )
        self.input_norm = nn.GroupNorm(num_groups=1, num_channels=hidden_channels)
        self.input_act = nn.GELU()

        self.blocks = nn.Sequential(
            ConvBlock(channels=hidden_channels, kernel_size=2, stride=2),
            ConvBlock(channels=hidden_channels, kernel_size=2, stride=2),
            ConvBlock(channels=hidden_channels, kernel_size=2, stride=2),
        )

    def forward(self, x):
        # x: [B, 12, 2500]
        x = self.input_proj(x)   # [B, 256, ~1250]
        x = self.input_norm(x)
        x = self.input_act(x)

        x = self.blocks(x)       # [B, 256, ~156]
        return x


class RelativePositionalConv(nn.Module):
    """
    Relative positional embeddings learned with a grouped convolution.
    Paper mentions 128 filters and 16 groups.
    Since transformer dim is 768, we project 768 -> 128 -> 768 through conv+proj.
    """
    def __init__(self, embed_dim=768, pos_dim=128, kernel_size=128, groups=16):
        super().__init__()
        self.in_proj = nn.Linear(embed_dim, pos_dim)
        self.conv = nn.Conv1d(
            in_channels=pos_dim,
            out_channels=pos_dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=groups
        )
        self.act = nn.GELU()
        self.out_proj = nn.Linear(pos_dim, embed_dim)

    def forward(self, x):
        # x: [B, T, D]
        pos = self.in_proj(x)          # [B, T, 128]
        pos = pos.transpose(1, 2)      # [B, 128, T]
        pos = self.conv(pos)           # [B, 128, T']
        pos = pos[..., :x.size(1)]     # trim in case padding adds 1 token
        pos = pos.transpose(1, 2)      # [B, T, 128]
        pos = self.act(pos)
        pos = self.out_proj(pos)       # [B, T, D]
        return x + pos


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim=768, num_heads=12, ff_dim=3072, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim)
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None, need_weights=False):
        # Self-attention
        h = self.norm1(x)
        attn_out, attn_weights = self.attn(
            h, h, h,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=need_weights,
            average_attn_weights=False
        )
        x = x + self.dropout1(attn_out)

        # Feed-forward
        h = self.norm2(x)
        ff_out = self.ff(h)
        x = x + self.dropout2(ff_out)

        if need_weights:
            return x, attn_weights
        return x


class ECGFMEncoder(nn.Module):
    """
    Full-size ECG-FM-style encoder:
    - CNN feature extractor
    - projection to transformer dim
    - relative positional conv
    - 12 transformer blocks
    """
    def __init__(
        self,
        in_channels=12,
        cnn_dim=256,
        embed_dim=768,
        num_heads=12,
        ff_dim=3072,
        num_layers=12,
        dropout=0.1,
        pos_dim=128,
        pos_kernel=128,
        pos_groups=16,
        global_pool="mean",
    ):
        super().__init__()

        self.feature_extractor = ECGFeatureExtractor(
            in_channels=in_channels,
            hidden_channels=cnn_dim
        )

        self.project_to_transformer = nn.Linear(cnn_dim, embed_dim)

        self.pos_conv = RelativePositionalConv(
            embed_dim=embed_dim,
            pos_dim=pos_dim,
            kernel_size=pos_kernel,
            groups=pos_groups
        )

        self.transformer_layers = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.global_pool = global_pool

    def forward(self, x, return_sequence=True, return_global=True, return_attn=False):
        """
        x: [B, 12, 2500]

        returns dict with:
          - z: latent CNN features before transformer, [B, T', 256]
          - c: contextualized features after transformer, [B, T', 768]
          - g: global pooled representation, [B, 768]
        """
        # CNN extractor
        z = self.feature_extractor(x)      # [B, 256, T']
        z = z.transpose(1, 2)              # [B, T', 256]

        # Project to transformer dim
        c = self.project_to_transformer(z) # [B, T', 768]

        # Relative positional embeddings
        c = self.pos_conv(c)

        attn_maps = []
        for layer in self.transformer_layers:
            if return_attn:
                c, attn = layer(c, need_weights=True)
                attn_maps.append(attn)
            else:
                c = layer(c)

        c = self.final_norm(c)

        out = {"z": z}

        if return_sequence:
            out["c"] = c

        if return_global:
            if self.global_pool == "mean":
                g = c.mean(dim=1)
            elif self.global_pool == "max":
                g = c.max(dim=1).values
            else:
                raise ValueError(f"Unsupported global_pool: {self.global_pool}")
            out["g"] = g

        if return_attn:
            out["attn_maps"] = attn_maps

        return out

## Forward Pass

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ECGFMEncoder().to(device)

x = torch.randn(2, 12, 2500).to(device)  # batch of 2 ECG segments
out = model(x)

print("z shape:", out["z"].shape)   # CNN latent reps
print("c shape:", out["c"].shape)   # transformer sequence reps
print("g shape:", out["g"].shape)   # pooled embedding

## Parameter Count

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable params: {count_params(model):,}")

## CMSC

In [ ]:
class ECGFMContrastiveModel(nn.Module):
    def __init__(self, encoder: nn.Module, proj_dim=256):
        super().__init__()
        self.encoder = encoder
        self.projection_head = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(),
            nn.Linear(768, proj_dim)
        )

    def forward(self, x1, x2):
        g1 = self.encoder(x1)["g"]
        g2 = self.encoder(x2)["g"]

        p1 = self.projection_head(g1)
        p2 = self.projection_head(g2)

        p1 = F.normalize(p1, dim=-1)
        p2 = F.normalize(p2, dim=-1)

        return p1, p2

## Loss

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.1):
    """
    z1, z2: [B, D], already normalized
    """
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # [2B, D]

    sim = torch.matmul(z, z.T) / temperature
    mask = torch.eye(2 * B, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(mask, -1e9)

    targets = torch.arange(B, device=z.device)
    targets = torch.cat([targets + B, targets], dim=0)

    loss = F.cross_entropy(sim, targets)
    return loss

In [ ]:
contrastive_model = ECGFMContrastiveModel(model).to(device)

x1 = torch.randn(2, 12, 2500).to(device)
x2 = torch.randn(2, 12, 2500).to(device)

z1, z2 = contrastive_model(x1, x2)
loss = nt_xent_loss(z1, z2)

print("z1:", z1.shape)
print("z2:", z2.shape)
print("loss:", loss.item())

## Backward Pass

In [ ]:
contrastive_model = ECGFMContrastiveModel(ECGFMEncoder().to(device)).to(device)
optimizer = torch.optim.Adam(contrastive_model.parameters(), lr=1e-4)

x1 = torch.randn(2, 12, 2500).to(device)
x2 = torch.randn(2, 12, 2500).to(device)

optimizer.zero_grad()
z1, z2 = contrastive_model(x1, x2)
loss = nt_xent_loss(z1, z2)
loss.backward()
optimizer.step()

print("loss:", loss.item())

## Test with 1 ECG Batch

In [ ]:
signal, meta = wfdb.rdsamp(record_path)
segments = preprocess_record(signal, meta["fs"])

In [ ]:
print("segments shape:", segments.shape)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

x1 = torch.tensor(segments[0], dtype=torch.float32).unsqueeze(0).to(device)  # [1, 12, 2500]
x2 = torch.tensor(segments[1], dtype=torch.float32).unsqueeze(0).to(device)  # [1, 12, 2500]

print("x1 shape:", x1.shape)
print("x2 shape:", x2.shape)

In [ ]:
## RUNNING ENCODER ONLY
model = ECGFMEncoder().to(device)
model.eval()

with torch.no_grad():
    out1 = model(x1)
    out2 = model(x2)

print("Encoder output for x1")
print("z:", out1["z"].shape)
print("c:", out1["c"].shape)
print("g:", out1["g"].shape)

print("\nEncoder output for x2")
print("z:", out2["z"].shape)
print("c:", out2["c"].shape)
print("g:", out2["g"].shape)

In [ ]:
contrastive_model = ECGFMContrastiveModel(ECGFMEncoder().to(device)).to(device)
contrastive_model.eval()

with torch.no_grad():
    z1, z2 = contrastive_model(x1, x2)

print("z1 shape:", z1.shape)
print("z2 shape:", z2.shape)

## Test with batch size 2 for contrastive loss

In [ ]:
record_paths = [
    os.path.join(PTB_DIR, f)
    for f in ptb_df["filename_hr"]
]

print(record_paths[:5])

In [ ]:
import wfdb

record_path = record_paths[0]

signal, meta = wfdb.rdsamp(record_path)

print(signal.shape)
print(meta["fs"])

In [ ]:
import random

record_path1 = random.choice(record_paths)
record_path2 = random.choice(record_paths)

print(record_path1)
print(record_path2)

In [ ]:
import torch
import wfdb

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load 2 real ECG records
signal1, meta1 = wfdb.rdsamp(record_path1)
signal2, meta2 = wfdb.rdsamp(record_path2)

segments1 = preprocess_record(signal1, meta1["fs"])
segments2 = preprocess_record(signal2, meta2["fs"])

print("segments1:", segments1.shape)
print("segments2:", segments2.shape)

# Make sure both have at least 2 segments
assert segments1.shape[0] >= 2
assert segments2.shape[0] >= 2

# Build real contrastive batch
x1_batch = torch.tensor(
    [segments1[0], segments2[0]],
    dtype=torch.float32
).to(device)

x2_batch = torch.tensor(
    [segments1[1], segments2[1]],
    dtype=torch.float32
).to(device)

print("x1_batch:", x1_batch.shape)
print("x2_batch:", x2_batch.shape)

# Model
contrastive_model = ECGFMContrastiveModel(ECGFMEncoder().to(device)).to(device)
optimizer = torch.optim.Adam(contrastive_model.parameters(), lr=1e-4)

# Forward + loss + backward
contrastive_model.train()
optimizer.zero_grad()

z1, z2 = contrastive_model(x1_batch, x2_batch)
loss = nt_xent_loss(z1, z2)

print("z1:", z1.shape)
print("z2:", z2.shape)
print("loss:", loss.item())

loss.backward()
optimizer.step()

print("Real ECG forward/backward pass successful.")

## Dataset + Dataloader

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
import wfdb


class ECGContrastiveDataset(Dataset):
    def __init__(self, record_paths, preprocess_fn, return_path=False):
        self.record_paths = record_paths
        self.preprocess_fn = preprocess_fn
        self.return_path = return_path

    def __len__(self):
        return len(self.record_paths)

    def __getitem__(self, idx):
        record_path = self.record_paths[idx]

        signal, meta = wfdb.rdsamp(record_path)
        segments = self.preprocess_fn(signal, meta["fs"])   # [n_segments, 12, 2500]

        if segments.shape[0] < 2:
            raise ValueError(f"Record {record_path} has fewer than 2 full segments")

        x1 = segments[0]
        x2 = segments[1]

        x1 = torch.from_numpy(np.asarray(x1)).float()
        x2 = torch.from_numpy(np.asarray(x2)).float()

        if self.return_path:
            return x1, x2, record_path

        return x1, x2

In [ ]:
dataset = ECGContrastiveDataset(
    record_paths=record_paths,
    preprocess_fn=preprocess_record,
    return_path=True
)

print("Dataset size:", len(dataset))

x1, x2, path = dataset[0]
print("record path:", path)
print("x1 shape:", x1.shape)
print("x2 shape:", x2.shape)

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

In [ ]:
batch = next(iter(loader))
x1, x2, paths = batch

print("x1 batch shape:", x1.shape)
print("x2 batch shape:", x2.shape)
print("first path:", paths[0])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

contrastive_model = ECGFMContrastiveModel(ECGFMEncoder().to(device)).to(device)
contrastive_model.eval()

x1, x2, paths = next(iter(loader))
x1 = x1.to(device, non_blocking=True)
x2 = x2.to(device, non_blocking=True)

with torch.no_grad():
    z1, z2 = contrastive_model(x1, x2)
    loss = nt_xent_loss(z1, z2)

print("x1:", x1.shape)
print("x2:", x2.shape)
print("z1:", z1.shape)
print("z2:", z2.shape)
print("loss:", loss.item())

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

contrastive_model = ECGFMContrastiveModel(ECGFMEncoder().to(device)).to(device)
optimizer = torch.optim.Adam(contrastive_model.parameters(), lr=1e-4)

contrastive_model.train()

for step, batch in enumerate(loader):
    x1, x2, paths = batch

    x1 = x1.to(device, non_blocking=True)
    x2 = x2.to(device, non_blocking=True)

    optimizer.zero_grad()

    z1, z2 = contrastive_model(x1, x2)
    loss = nt_xent_loss(z1, z2)

    loss.backward()
    optimizer.step()

    print(f"step={step} loss={loss.item():.4f}")

    if step == 4:
        break

In [ ]:
contrastive_model.train()
running_loss = 0.0

for step, (x1, x2, paths) in enumerate(loader):
    x1 = x1.to(device, non_blocking=True)
    x2 = x2.to(device, non_blocking=True)

    optimizer.zero_grad()

    z1, z2 = contrastive_model(x1, x2)
    loss = nt_xent_loss(z1, z2)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    if (step + 1) % 10 == 0:
        avg_loss = running_loss / 10
        print(f"step={step+1} avg_loss={avg_loss:.4f}")
        running_loss = 0.0

    if step == 99:
        break

## Adding PTB-XL + Arrhythmia Study Dataset.

In [ ]:
ARR_PATH = r"a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0"

In [ ]:
from pathlib import Path

hea_files = list(Path(ARR_PATH).rglob("*.hea"))

print("Total records:", len(hea_files))
print("Example:", hea_files[:5])

In [ ]:
arr_record_paths = [str(p.with_suffix("")) for p in hea_files]

print(arr_record_paths[:5])

In [ ]:
import wfdb

signal, meta = wfdb.rdsamp(arr_record_paths[0])

print("shape:", signal.shape)
print("fs:", meta["fs"])
print("leads:", meta["sig_name"])

In [ ]:
all_records = []

# PTB-XL
for p in record_paths:
    all_records.append({
        "dataset": "ptbxl",
        "record_path": p
    })

# Arrhythmia
for p in arr_record_paths:
    all_records.append({
        "dataset": "arrhythmia",
        "record_path": p
    })

print("Total records:", len(all_records))
print(all_records[:3])

In [ ]:
print("PTB example:", record_paths[0])
print("Arr example:", arr_record_paths[0])

In [ ]:
wfdb.rdsamp(record_paths[0])
wfdb.rdsamp(arr_record_paths[0])

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset
import wfdb
import random


class ECGContrastiveDataset(Dataset):
    def __init__(self, records, preprocess_fn, return_info=False, max_retries=10):
        self.records = records
        self.preprocess_fn = preprocess_fn
        self.return_info = return_info
        self.max_retries = max_retries

    def __len__(self):
        return len(self.records)

    def _load_item(self, idx):
        item = self.records[idx]
        dataset_name = item["dataset"]
        record_path = item["record_path"]

        signal, meta = wfdb.rdsamp(record_path)
        segments = self.preprocess_fn(signal, meta["fs"])

        if segments.shape[0] < 2:
            raise ValueError(f"{dataset_name} record {record_path} has fewer than 2 segments")

        x1 = torch.from_numpy(np.asarray(segments[0])).float()
        x2 = torch.from_numpy(np.asarray(segments[1])).float()

        if self.return_info:
            return x1, x2, dataset_name, record_path

        return x1, x2

    def __getitem__(self, idx):
        for _ in range(self.max_retries):
            try:
                return self._load_item(idx)
            except Exception:
                idx = random.randint(0, len(self.records) - 1)

        raise RuntimeError("Failed to load valid ECG sample")

In [ ]:
from torch.utils.data import DataLoader

dataset = ECGContrastiveDataset(
    records=all_records,
    preprocess_fn=preprocess_record,
    return_info=True
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

In [ ]:
x1, x2, dataset_names, record_paths = next(iter(loader))

print("x1:", x1.shape)
print("x2:", x2.shape)
print("datasets in batch:", dataset_names[:10])
print("example path:", record_paths[0])

In [ ]:
import os
import torch

SAVE_DIR = r"C:\Users\eq26\Desktop\ECG LM"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_checkpoint(path, model, optimizer, epoch, step, loss_history):
    ckpt = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "step": step,
        "loss_history": loss_history,
    }
    torch.save(ckpt, path)
    print(f"Saved checkpoint: {path}")

def load_checkpoint(path, model, optimizer=None, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    epoch = ckpt.get("epoch", 0)
    step = ckpt.get("step", 0)
    loss_history = ckpt.get("loss_history", [])

    print(f"Loaded checkpoint from {path}")
    return epoch, step, loss_history

In [ ]:
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = ECGFMEncoder().to(device)
model   = ECGFMContrastiveModel(encoder).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_epochs       = 10
print_every      = 10
save_every_epochs = 1
MAX_GRAD_NORM    = 1.0

loss_history = []
global_step  = 0
start_epoch  = 0

RESUME_PATH = None
if RESUME_PATH is not None and os.path.exists(RESUME_PATH):
    start_epoch, global_step, loss_history = load_checkpoint(
        RESUME_PATH, model, optimizer, device=device
    )

best_val_loss = float("inf")

for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss = 0.0
    epoch_losses = []
    nan_skips    = 0
    t0 = time.time()

    for step, batch in enumerate(loader):
        x1, x2, dataset_names, _ = batch

        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)

        # Skip batches with corrupt data before they hit the model
        if not (torch.isfinite(x1).all() and torch.isfinite(x2).all()):
            nan_skips += 1
            continue

        optimizer.zero_grad(set_to_none=True)

        z1, z2 = model(x1, x2)
        loss   = nt_xent_loss(z1, z2, temperature=0.1)

        if not torch.isfinite(loss):
            nan_skips += 1
            continue

        loss.backward()
        # Gradient clipping prevents exploding gradients in the Transformer
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

        loss_val = loss.item()
        running_loss += loss_val
        epoch_losses.append(loss_val)
        global_step += 1

        if (step + 1) % print_every == 0:
            avg = running_loss / print_every
            loss_history.append(avg)
            print(
                f"epoch={epoch+1}/{num_epochs} "
                f"step={step+1}/{len(loader)} "
                f"avg_loss={avg:.4f} "
                f"nan_skips={nan_skips}"
            )
            running_loss = 0.0

    mean_loss = sum(epoch_losses) / len(epoch_losses) if epoch_losses else float("nan")
    elapsed   = (time.time() - t0) / 60
    print(f"Epoch {epoch+1} done | mean_loss={mean_loss:.4f} | "
          f"nan_skips={nan_skips} | {elapsed:.1f}min")

    if (epoch + 1) % save_every_epochs == 0:
        ckpt_path = os.path.join(SAVE_DIR, f"ecgfm_epoch_{epoch+1}.pt")
        save_checkpoint(
            ckpt_path, model, optimizer,
            epoch=epoch + 1, step=global_step, loss_history=loss_history
        )

print("Pretraining complete.")
